# Notebook 3 — Solutions: Building the Connectome Graph

**BINFX410 · Chapter 10 · Connectomics**

Reference solutions for the five exercises in `03_connectome_graph.ipynb`.  
Run notebooks 1–3 first to generate `../data/connectome_graphs.pkl`.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx

DATA_DIR = Path('../data')

with open(DATA_DIR / 'connectome_graphs.pkl', 'rb') as f:
    data = pickle.load(f)

G_gt          = data['G_gt']
G_detected    = data['G_detected']
detected_soma = data['detected_soma']
gt_neurons    = data['gt_neurons']
gt_synapses   = data['gt_synapses']
image         = data['image']

with open(DATA_DIR / 'segmentation_results.pkl', 'rb') as f:
    seg = pickle.load(f)
detected_synapses = seg['detected_synapses']

print(f'Detected graph: {G_detected.number_of_nodes()} nodes, {G_detected.number_of_edges()} edges')
print(f'GT graph      : {G_gt.number_of_nodes()} nodes, {G_gt.number_of_edges()} edges')

---
## Exercise 3.1 — Edge Matching Accuracy vs. Ground Truth

**Task:** Compute the fraction of edges in `G_detected` that match the ground-truth edges in `G_gt`. What errors does the nearest-neighbour heuristic introduce?

In [ ]:
# To compare detected edges to GT edges we need to first establish a node correspondence
# between the detected soma and the GT neurons (they may be ordered differently).

# ── Step 1: Match detected soma to GT neurons by nearest centroid ─────────────
detected_arr = np.array(detected_soma)         # (N_det, 2)
gt_centers   = np.array([(n.soma_center[0], n.soma_center[1]) for n in gt_neurons])  # (N_gt, 2)

from scipy.spatial.distance import cdist
dist_matrix = cdist(detected_arr, gt_centers)  # (N_det, N_gt)

# Greedy matching: for each detected soma, find closest GT neuron
det_to_gt = {}
used_gt   = set()
for det_id in np.argsort(dist_matrix.min(axis=1)):
    gt_id = int(np.argmin(dist_matrix[det_id]))
    if gt_id not in used_gt and dist_matrix[det_id, gt_id] < 40:  # 40px tolerance
        det_to_gt[det_id] = gt_id
        used_gt.add(gt_id)

print('Detected → GT soma correspondence:')
for d, g in sorted(det_to_gt.items()):
    print(f'  Detected N{d} → GT neuron {g}  (dist={dist_matrix[d, g]:.1f}px)')

# ── Step 2: Convert detected edges to GT node space ───────────────────────────
detected_edges_in_gt = set()
for u, v in G_detected.edges():
    if u in det_to_gt and v in det_to_gt:
        detected_edges_in_gt.add((det_to_gt[u], det_to_gt[v]))

gt_edges = set(G_gt.edges())

true_positives  = detected_edges_in_gt & gt_edges
false_positives = detected_edges_in_gt - gt_edges
false_negatives = gt_edges - detected_edges_in_gt

precision = len(true_positives) / (len(detected_edges_in_gt) + 1e-8)
recall    = len(true_positives) / (len(gt_edges) + 1e-8)
f1        = 2 * precision * recall / (precision + recall + 1e-8)

print(f'\nEdge matching results:')
print(f'  Ground-truth edges   : {len(gt_edges)}')
print(f'  Detected edges (mapped): {len(detected_edges_in_gt)}')
print(f'  True positives       : {len(true_positives)}')
print(f'  False positives      : {len(false_positives)}  (wrong connections)')
print(f'  False negatives      : {len(false_negatives)}  (missed connections)')
print(f'  Precision            : {precision:.3f}')
print(f'  Recall               : {recall:.3f}')
print(f'  F1                   : {f1:.3f}')
print()
print('Sources of error in the nearest-neighbour heuristic:')
print('  1. Pre/post assignment: using proximity to determine which soma is pre- vs post-synaptic')
print('     can be wrong when axons of different neurons overlap in the same region.')
print('  2. Missing synapses: if the CNN fails to detect a synapse, that edge is lost entirely.')
print('  3. False synapse detections: noise near soma boundaries creates phantom edges.')

---
## Exercise 3.2 — `max_dist` Threshold Sweep

**Task:** Vary `max_dist` from 40 to 120. Plot edge count vs. threshold and explain the trade-off.

In [ ]:
def assign_synapses_to_neurons(soma_centroids, synapse_centroids, max_dist=80.0):
    soma_arr = np.array(soma_centroids)
    edges = []
    for syn in synapse_centroids:
        syn_arr = np.array(syn)
        dists   = np.linalg.norm(soma_arr - syn_arr, axis=1)
        order   = np.argsort(dists)
        if len(order) < 2: continue
        if dists[order[0]] < max_dist:
            edges.append({'pre_id': int(order[1]), 'post_id': int(order[0]),
                          'syn_r': syn[0], 'syn_c': syn[1], 'dist': float(dists[order[0]])})
    return edges

thresholds = np.arange(20, 141, 10)
edge_counts = []

for thresh in thresholds:
    edges = assign_synapses_to_neurons(detected_soma, detected_synapses, max_dist=thresh)
    # Build graph to count unique edges
    G_tmp = nx.DiGraph()
    G_tmp.add_nodes_from(range(len(detected_soma)))
    for e in edges:
        G_tmp.add_edge(e['pre_id'], e['post_id'])
    edge_counts.append(G_tmp.number_of_edges())

gt_edge_count = G_gt.number_of_edges()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, edge_counts, 'o-', color='steelblue', markersize=7, linewidth=2)
ax.axhline(gt_edge_count, color='tomato', linestyle='--', linewidth=2, label=f'GT edges ({gt_edge_count})')
ax.set_xlabel('max_dist threshold (pixels)', fontsize=12)
ax.set_ylabel('Number of unique edges in G_detected', fontsize=12)
ax.set_title('Exercise 3.2 — Edge Count vs. max_dist Threshold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Trade-off explanation:')
print('  Low max_dist  → few edges assigned; high precision but low recall.')
print('    Real connections that are spatially spread are missed.')
print()
print('  High max_dist → many edges assigned; high recall but low precision.')
print('    Spurious long-range connections are created between distant neurons')
print('    just because their soma and a random synapse are close enough.')
print()
print('  Optimal threshold: the value nearest to GT edge count on the plateau.')
print('  Typically ~80–100px for this dataset, corresponding to roughly 1 soma radius distance.')

---
## Exercise 3.3 — Community Detection with Greedy Modularity

**Task:** Use `nx.community.greedy_modularity_communities`. How many communities? Are they spatially clustered?

In [ ]:
from networkx.algorithms import community

# Greedy modularity requires an undirected graph
G_undet = G_detected.to_undirected()

if G_undet.number_of_edges() == 0:
    print('No edges in undirected graph — cannot detect communities.')
else:
    communities = list(community.greedy_modularity_communities(G_undet))
    print(f'Number of communities found: {len(communities)}')
    for i, c in enumerate(communities):
        print(f'  Community {i}: neurons {sorted(c)}')

    # Map neuron → community index
    node_community = {}
    for ci, c in enumerate(communities):
        for node in c:
            node_community[node] = ci

    # Visualize: nodes colored by community, overlaid on the image
    cmap   = plt.cm.get_cmap('tab10', len(communities))
    colors = [cmap(node_community.get(n, 0)) for n in G_detected.nodes()]

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Graph layout
    pos = nx.get_node_attributes(G_detected, 'pos')
    if not pos:
        pos = nx.spring_layout(G_detected, seed=42)

    nx.draw_networkx(G_detected, pos=pos, ax=axes[0],
                     node_color=colors, node_size=400,
                     edge_color='gray', arrows=True, arrowsize=12,
                     labels={n: f'N{n}' for n in G_detected.nodes()},
                     font_size=9)
    axes[0].set_title('Communities in Detected Connectome\n(colors = community membership)', fontsize=12)
    axes[0].axis('off')

    # Overlay on image
    axes[1].imshow(image, cmap='gray', vmin=0, vmax=0.8)
    for i, (r, c) in enumerate(detected_soma):
        comm_color = cmap(node_community.get(i, 0))
        circle = plt.Circle((c, r), 18, fill=True, facecolor=comm_color,
                             edgecolor='white', alpha=0.7, linewidth=1.5)
        axes[1].add_patch(circle)
        axes[1].text(c, r, f'N{i}', ha='center', va='center', fontsize=8,
                     color='black', fontweight='bold')
    axes[1].set_title('Communities overlaid on fluorescence image', fontsize=12)
    axes[1].axis('off')

    plt.suptitle('Exercise 3.3 — Community Detection (Greedy Modularity)', fontsize=14)
    plt.tight_layout()
    plt.show()

    print('\nDoes spatial clustering match communities?')
    print('  Neurons with direct connections tend to be placed nearby in our synthetic image.')
    print('  Greedy modularity optimizes edge density within communities, so neurons')
    print('  connected by short axons (which form more synapses) will cluster together.')
    print('  Whether this matches spatial proximity depends on the specific scene seed.')

---
## Exercise 3.4 — Is the Connectome a Small-World Network?

**Task:** Compute clustering coefficient and average path length. Compare to a random graph of the same size.

In [ ]:
N_nodes = G_detected.number_of_nodes()
N_edges = G_detected.number_of_edges()

G_undet = G_detected.to_undirected()

# Handle disconnected graph by using the largest connected component
largest_cc    = max(nx.connected_components(G_undet), key=len)
G_lcc         = G_undet.subgraph(largest_cc).copy()

if len(G_lcc) >= 3 and G_lcc.number_of_edges() > 0:
    C_connectome = nx.average_clustering(G_lcc)
    L_connectome = nx.average_shortest_path_length(G_lcc)
else:
    C_connectome = 0.0
    L_connectome = float('nan')

# Generate random graphs with same n and m; average over 50 trials
n_rand_trials = 50
C_rand_list, L_rand_list = [], []

for seed in range(n_rand_trials):
    Gr    = nx.gnm_random_graph(N_nodes, N_edges, seed=seed)
    lcc_r = max(nx.connected_components(Gr), key=len)
    Gr_lcc = Gr.subgraph(lcc_r).copy()
    if len(Gr_lcc) >= 3 and Gr_lcc.number_of_edges() > 0:
        try:
            C_rand_list.append(nx.average_clustering(Gr_lcc))
            L_rand_list.append(nx.average_shortest_path_length(Gr_lcc))
        except nx.NetworkXError:
            pass

C_rand = np.mean(C_rand_list) if C_rand_list else 0
L_rand = np.mean(L_rand_list) if L_rand_list else float('nan')

print('Small-World Analysis')
print('=' * 50)
print(f'  Graph size: {N_nodes} nodes, {N_edges} edges')
print(f'  Largest CC: {len(largest_cc)} nodes')
print()
print(f'  Clustering coefficient:')
print(f'    Connectome : {C_connectome:.4f}')
print(f'    Random     : {C_rand:.4f}  (mean over {n_rand_trials} trials)')
print(f'    Ratio C/C_r: {C_connectome/(C_rand+1e-8):.2f}x')
print()
print(f'  Average path length:')
print(f'    Connectome : {L_connectome:.4f}')
print(f'    Random     : {L_rand:.4f}')
print(f'    Ratio L/L_r: {L_connectome/(L_rand+1e-8):.2f}x')

# Small-world coefficient sigma = (C/C_r) / (L/L_r)
if L_rand > 0 and C_rand > 0:
    sigma = (C_connectome / C_rand) / (L_connectome / L_rand)
    print(f'\n  Small-world coefficient σ = (C/C_r)/(L/L_r) = {sigma:.3f}')
    print(f'  σ > 1 indicates small-world topology.')
    if sigma > 1:
        print(f'  CONCLUSION: The connectome shows SMALL-WORLD properties.')
    else:
        print(f'  CONCLUSION: The connectome does NOT show strong small-world properties.')
        print(f'    (Note: with only {N_nodes} nodes and {N_edges} edges, graph statistics are noisy.)')

print()
print('Interpretation:')
print('  Small-world networks have high local clustering (neighbors of neighbors are')
print('  often connected) yet short global path lengths (any node reachable in few hops).')
print('  This topology supports efficient local processing AND global information integration.')
print('  Real connectomes (e.g., C. elegans) are canonical small-world networks.')

---
## Exercise 3.5 (Challenge) — Connectome Precision and Recall

**Task:** Implement `connectome_precision_recall(G_detected, G_gt)` with proper node matching.

In [ ]:
from scipy.spatial.distance import cdist as scipy_cdist

def connectome_precision_recall(
    G_detected, detected_soma_list,
    G_gt, gt_neurons_list,
    node_match_dist_threshold=40.0,
):
    """
    Compute precision and recall for connectome edge detection.

    Parameters
    ----------
    G_detected            : networkx DiGraph — detected connectome
    detected_soma_list    : list of (r, c) centroids for detected soma nodes
    G_gt                  : networkx DiGraph — ground-truth connectome
    gt_neurons_list       : list of Neuron objects with .soma_center attribute
    node_match_dist_threshold : max pixel distance to consider two soma the same neuron

    Returns
    -------
    dict with precision, recall, f1, true_positives, false_positives, false_negatives
    """
    # ── Node correspondence via nearest-neighbour matching ────────────────────
    det_arr = np.array(detected_soma_list)
    gt_arr  = np.array([(n.soma_center[0], n.soma_center[1]) for n in gt_neurons_list])

    D = scipy_cdist(det_arr, gt_arr)

    det_to_gt = {}
    used = set()
    # Sort detected nodes by their minimum distance to any GT node
    for det_id in np.argsort(D.min(axis=1)):
        gt_id = int(np.argmin(D[det_id]))
        if gt_id not in used and D[det_id, gt_id] <= node_match_dist_threshold:
            det_to_gt[det_id] = gt_id
            used.add(gt_id)

    n_matched_nodes = len(det_to_gt)

    # ── Translate detected edges into GT-node space ───────────────────────────
    detected_in_gt_space = set()
    for u, v in G_detected.edges():
        if u in det_to_gt and v in det_to_gt:
            detected_in_gt_space.add((det_to_gt[u], det_to_gt[v]))

    gt_edge_set = set(G_gt.edges())

    TP = detected_in_gt_space & gt_edge_set
    FP = detected_in_gt_space - gt_edge_set
    FN = gt_edge_set - detected_in_gt_space

    precision = len(TP) / (len(TP) + len(FP) + 1e-8)
    recall    = len(TP) / (len(TP) + len(FN) + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return {
        'precision'      : precision,
        'recall'         : recall,
        'f1'             : f1,
        'true_positives' : sorted(TP),
        'false_positives': sorted(FP),
        'false_negatives': sorted(FN),
        'n_detected_edges': G_detected.number_of_edges(),
        'n_gt_edges'      : G_gt.number_of_edges(),
        'n_matched_nodes' : n_matched_nodes,
        'n_det_nodes'     : G_detected.number_of_nodes(),
        'n_gt_nodes'      : G_gt.number_of_nodes(),
    }


metrics = connectome_precision_recall(
    G_detected, detected_soma,
    G_gt, gt_neurons,
)

print('Connectome Precision/Recall Report')
print('=' * 45)
print(f'  Node matching  : {metrics["n_matched_nodes"]} / {metrics["n_gt_nodes"]} GT neurons matched')
print(f'  Detected edges : {metrics["n_detected_edges"]}')
print(f'  GT edges       : {metrics["n_gt_edges"]}')
print()
print(f'  True positives : {len(metrics["true_positives"])}  {metrics["true_positives"]}')
print(f'  False positives: {len(metrics["false_positives"])} {metrics["false_positives"]}')
print(f'  False negatives: {len(metrics["false_negatives"])} {metrics["false_negatives"]}')
print()
print(f'  Precision : {metrics["precision"]:.3f}')
print(f'  Recall    : {metrics["recall"]:.3f}')
print(f'  F1        : {metrics["f1"]:.3f}')
print()
print('Sources of error:')
print('  FP edges: phantom synapses detected by CNN assigned to wrong neuron pairs.')
print('  FN edges: real synapses missed by CNN (especially small/faint ones).')
print('  Node mis-matching: if detected soma drifts >40px from GT, edges are uncountable.')